In [51]:
import numpy as mp
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import RandomForestClassifier
import joblib
from google.colab import files

In [52]:
dataset = pd.read_csv('/content/sample_data/dead_stock_training_data.csv')

In [53]:
dataset.head()

,store_id,product_id,current_stock,weekly_sales_rate,expiry_date,product_price,weeks_to_expiry,label
0,S013,P021,152,79.86,2025-07-20,75.88,3.00,1
1,S020,P015,750,60.09,2025-09-26,19.00,12.71,1
2,S014,P008,508,86.75,2025-07-29,68.58,4.29,1
3,S003,P012,358,97.02,2025-08-04,29.11,5.14,1
4,S003,P020,241,99.23,2025-08-07,37.38,5.57,0


In [54]:
dataset.shape

(200, 8)

In [55]:
dataset['label'].value_counts()

,count
label,
1,170
0,30


In [56]:
dataset.select_dtypes(include='number').groupby('label').mean()

,current_stock,weekly_sales_rate,product_price,weeks_to_expiry
label,,,,
0,269.000000,74.017000,50.010667,9.171333
1,579.935294,48.879294,55.901882,5.785588


In [57]:
X = dataset.drop(columns = 'label' ,axis = 1)
Y = dataset['label']

In [58]:
print(X)
print(Y)

    store_id product_id  current_stock  weekly_sales_rate expiry_date  \
0       S013       P021            152              79.86  2025-07-20   
1       S020       P015            750              60.09  2025-09-26   
2       S014       P008            508              86.75  2025-07-29   
3       S003       P012            358              97.02  2025-08-04   
4       S003       P020            241              99.23  2025-08-07   
..       ...        ...            ...                ...         ...   
195     S009       P017            282              50.35  2025-08-08   
196     S018       P014            985              43.73  2025-09-26   
197     S006       P017            662              36.95  2025-07-31   
198     S020       P018            492              36.25  2025-09-13   
199     S010       P025            338               1.75  2025-07-26   

     product_price  weeks_to_expiry  
0            75.88             3.00  
1            19.00            12.71  
2        

In [59]:
X = X.drop(columns='expiry_date')
# expiry_date is a string (like "2025-08-15"), so it's not usable in a machine learning model as-is.

# already created weeks_to_expiry, which is a numeric version and more useful.

In [64]:
X = pd.get_dummies(X, columns=['store_id', 'product_id'])
# # Columns like store_id and product_id are categorical (e.g., "S005", "P013")

# We convert them into binary columns using one-hot encoding

In [65]:
print(X.columns)

Index(['current_stock', 'weekly_sales_rate', 'product_price',
       'weeks_to_expiry', 'store_id_S001', 'store_id_S002', 'store_id_S003',
       'store_id_S004', 'store_id_S005', 'store_id_S006', 'store_id_S007',
       'store_id_S008', 'store_id_S009', 'store_id_S010', 'store_id_S011',
       'store_id_S012', 'store_id_S013', 'store_id_S014', 'store_id_S015',
       'store_id_S016', 'store_id_S017', 'store_id_S018', 'store_id_S019',
       'store_id_S020', 'product_id_P001', 'product_id_P002',
       'product_id_P003', 'product_id_P004', 'product_id_P005',
       'product_id_P006', 'product_id_P007', 'product_id_P008',
       'product_id_P009', 'product_id_P010', 'product_id_P011',
       'product_id_P012', 'product_id_P013', 'product_id_P014',
       'product_id_P015', 'product_id_P016', 'product_id_P017',
       'product_id_P018', 'product_id_P019', 'product_id_P020',
       'product_id_P021', 'product_id_P022', 'product_id_P023',
       'product_id_P024', 'product_id_P025', 'produ

In [66]:
X_train , X_test , Y_train , Y_test= train_test_split(X,Y,test_size=0.2,stratify=Y,random_state=42)

In [67]:
model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',   # Handles class imbalance
    random_state=42
)

model.fit(X_train, Y_train)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [68]:
Y_pred = model.predict(X_test)


In [69]:
print("Confusion Matrix:\n", confusion_matrix(Y_test, Y_pred))
print("\nClassification Report:\n", classification_report(Y_test, Y_pred))
print("\nAccuracy Score:", accuracy_score(Y_test, Y_pred))


Confusion Matrix:
 [[ 5  1]
 [ 0 34]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.83      0.91         6
           1       0.97      1.00      0.99        34

    accuracy                           0.97        40
   macro avg       0.99      0.92      0.95        40
weighted avg       0.98      0.97      0.97        40


Accuracy Score: 0.975


In [70]:
joblib.dump(model, 'dead_stock_model.pkl')


['dead_stock_model.pkl']

In [71]:
files.download('dead_stock_model.pkl')



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [72]:
sample_input = {
    'current_stock': [500],
    'weekly_sales_rate': [12.5],
    'product_price': [55.0],
    'weeks_to_expiry': [2.3],
    'store_id_S003': [1],
    'product_id_P008': [1]
}

In [73]:
input_df = pd.DataFrame(sample_input)


In [74]:
input_df = input_df.reindex(columns=model.feature_names_in_, fill_value=0)


In [76]:
prediction = model.predict(input_df)[0]

In [77]:
result = "Dead Stock" if prediction == 1 else "Not Dead Stock"
print(f"Prediction: {result}")


Prediction: Dead Stock
